# 08 — Thesis & Paper Figure Reproduction

**Purpose:** Reproduce the key figures from the PhD thesis (Chapters 4–6) and the published paper on recursive prototyping (Chapter 5).

**Status:** Complete — all code cells implemented and ported from thesis-era notebook.

**Source:** `smartflat-thesis-local/notebooks/thesis-content/thesis-figures.ipynb` (figures for Ch. 5 + paper); Ch. 4 and Ch. 6 figures written from scratch using codebase utilities.

**Requirements:**
- Data drive mounted (`SMARTFLAT_DATA_ROOT` set)
- All feature extraction outputs available (VideoMAE-v2, prototypes, segments)
- Computed symbolization results in `{DATA_ROOT}/outputs/` and `{DATA_ROOT}/experiments/`
- Manual annotation constants in `smartflat.constants_annotations_prototypes` (required for prototype coloring)

**Note:** Some figures depend on intermediate results from the recursive prototyping pipeline (NB03–NB05) which require prior human annotation. These cannot be regenerated from scratch — they rely on existing computed outputs. See Section 2 for annotation dependency details.

## 0. Setup

In [ ]:
import os

# Set data root to thesis-era data location
os.environ['SMARTFLAT_DATA_ROOT'] = '/Users/samperochon/github-repositories/smartflat-thesis-local/data/data-gold-final'


In [ ]:
#%pip install -e ..

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import umap
import joblib
from collections import Counter
from sklearn.preprocessing import StandardScaler

# --- Setup and data loading ---
from smartflat.utils.utils_io import get_data_root, load_df
from smartflat.configs.loader import import_config
from smartflat.constants import incomplete_clinical_administrations

# --- Annotation constants (manual annotation outputs) ---
from smartflat.constants_annotations_prototypes import mapping_cluster_category, mapping_category_df

# --- Symbolization main pipeline ---
from smartflat.features.symbolization.main import define_symbolization, build_clusterdf
from smartflat.features.symbolization.utils_dataset import get_experiments_dataframe
from smartflat.features.symbolization.utils import (
    fix_clinical_diagnosis,
    add_pyramid_labels,
    update_segmentation_from_embedding_labels,
    get_prototypes,
    compute_cluster_probabilities,
)

# --- Visualization utilities ---
from smartflat.utils.utils_visualization import (
    plot_chronogames,
    plot_gram,
    plot_per_cat_x_cont_y_distributions,
)
from smartflat.features.symbolization.visualization import (
    plot_cluster_prevalence,
    plot_labels_counts_per_diagnosis,
)
from smartflat.features.symbolic_barycenter.visualization import (
    plot_pairwise_twe_distances_by_group,
    plot_distance_violin,
)

# --- Dataset utilities ---
from smartflat.datasets.utils import load_embeddings, load_embedding_dimensions, add_umap
from smartflat.utils.utils_dataset import normalize_data


In [ ]:
DATA_ROOT = get_data_root()
DATA_AVAILABLE = DATA_ROOT is not None and os.path.exists(DATA_ROOT)

print(f"Data root: {DATA_ROOT}")
print(f"Data available: {DATA_AVAILABLE}")

if not DATA_AVAILABLE:
    print("WARNING: Data drive not mounted. Figure generation requires computed outputs.")
else:
    # Figure output directory (new convention, not used in NB01–NB07)
    fig_dir = os.path.join(DATA_ROOT, "figures", "thesis")
    os.makedirs(fig_dir, exist_ok=True)
    print(f"Figure output directory: {fig_dir}")
    
# Configuration constants (for all figure sections)
ANNOTATOR_ID = 'samperochon'
ROUND_NUMBER = 8
CONFIG_NAME = 'SymbolicSourceInferenceGoldConfig'
TASK_NAME = 'cuisine'
MODALITY = 'Tobii'


## 1. Chapter 4 — Study Design & Data Preprocessing

Multimodal recording setup and cohort composition.

**Pipeline overview:**
1. Load symbolization registration dataframe (primary data source for all sections)
2. Apply clinical filters (diagnosis groups, exclude incomplete administrations)
3. Generate cohort statistics and duration distributions

**See:** `smartflat.constants`, `smartflat.features.symbolization.utils`

In [ ]:
if DATA_AVAILABLE:
    # Load the symbolization registration dataframe (primary data source for all sections)
    symb_pkl = os.path.join(
        DATA_ROOT, 'outputs', 'symbolization-gold', 'faissc_inference_symbolization',
        ANNOTATOR_ID, f'round_{ROUND_NUMBER}',
        f'{CONFIG_NAME}_symbolization_registration_dataframe.pkl'
    )
    
    if not os.path.exists(symb_pkl):
        print(f"ERROR: Symbolization dataframe not found at {symb_pkl}")
        print("Figure generation requires computed symbolization outputs.")
        df = None
    else:
        df = load_df(symb_pkl)
        print(f"Loaded symbolization dataframe: {df.shape[0]} rows")
        
        # Apply clinical filters
        df = fix_clinical_diagnosis(df)
        df = df[~df['participant_id'].isin(incomplete_clinical_administrations.keys())]
        df.sort_values('task_number_int', ascending=True, inplace=True)
        df = df[df['pathologie'].isin(['HEALTHY', 'RIL', 'TBI'])]
        df = df.drop_duplicates(subset='trigram', keep='first')
        
        print(f"After filtering: {df.shape[0]} unique administrations")
        print(f"Cohort breakdown:\n{df['pathologie'].value_counts()}")
else:
    print("Data not available. Skipping Chapter 4 figures.")


In [ ]:
if DATA_AVAILABLE and df is not None:
    # Fig 4.1: Cohort breakdown bar chart
    counts = df['pathologie'].value_counts().reindex(['HEALTHY', 'RIL', 'TBI'])
    fig, ax = plt.subplots(figsize=(5, 4))
    counts.plot.bar(ax=ax, color=['steelblue', 'tomato', 'goldenrod'])
    ax.set_xlabel('Diagnostic group', fontweight='bold')
    ax.set_ylabel('N participants (deduplicated)', fontweight='bold')
    ax.set_title('Cohort composition (N=122 unique participants)', fontweight='bold')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
    plt.tight_layout()
    fig.savefig(os.path.join(fig_dir, 'fig4_cohort_breakdown.png'), dpi=150, bbox_inches='tight')
    plt.show()


In [ ]:
if DATA_AVAILABLE and df is not None:
    # Fig 4.2: Task duration distribution by diagnostic group
    fig, axes = plt.subplots(figsize=(8, 5))
    sns.boxplot(data=df, x='pathologie', y='duration', ax=axes, palette=['steelblue', 'tomato', 'goldenrod'], order=['HEALTHY', 'TBI', 'RIL'])
    axes.set_xlabel('Diagnostic group', fontweight='bold')
    axes.set_ylabel('Task duration (seconds)', fontweight='bold')
    axes.set_title('Recording duration by diagnostic group', fontweight='bold')
    plt.tight_layout()
    fig.savefig(os.path.join(fig_dir, 'fig4_duration_by_group.png'), dpi=150, bbox_inches='tight')
    plt.show()


## 2. Chapter 5 — UMAP Visualization of Latent Space

UMAP projection of VideoMAE-v2 embeddings showing prototypes and empirical data.

**Pipeline overview:**
1. Load prototype cluster centers and full embedding matrix
2. Apply StandardScaler normalization + L2 normalization
3. Fit UMAP jointly on prototypes and empirical data
4. Color prototypes by semantic category (requires manual annotation)

**See:** `smartflat.features.symbolization.utils`, `smartflat.datasets.utils`

**Annotation dependency:** Requires `mapping_cluster_category` from `smartflat.constants_annotations_prototypes` — the output of manual prototype labeling (annotation round 8). The UMAP coloring by category will be incorrect without the correct annotation constants.

In [ ]:
if DATA_AVAILABLE and df is not None:
    try:
        # Load prototype cluster centers
        P_path = os.path.join(
            DATA_ROOT, 'experiments', 'symbolization-gold', 'faissc_inference',
            ANNOTATOR_ID, f'round_{ROUND_NUMBER}',
            f'{TASK_NAME}_{MODALITY}_cluster_centers.npy'
        )
        P = np.load(P_path)
        print(f"Loaded prototypes: shape {P.shape}")
        
        # Load training scaler (used for normalization)
        scaler_path = os.path.join(
            DATA_ROOT, 'experiments', 'symbolization-gold', 'faissc_refinement',
            ANNOTATOR_ID, f'round_{ROUND_NUMBER}',
            'training_scaler_N_725335.pkl'
        )
        if os.path.exists(scaler_path):
            scaler = joblib.load(scaler_path)
            print(f"Loaded StandardScaler from {scaler_path}")
        else:
            print(f"WARNING: Scaler not found at {scaler_path}. Using identity scaling.")
            scaler = None
        
        # Apply pyramid labels to get category assignments
        df = add_pyramid_labels(df, CONFIG_NAME, ANNOTATOR_ID, ROUND_NUMBER)
        print(f"Applied pyramid labels to dataframe")
        
        # Create category labels for each frame sequence
        df['category_labels'] = df['pyr_filtered_raw_embedding_labels'].apply(
            lambda x: np.array([mapping_cluster_category.get(i, ['-1'])[0] if i != '-1' else '-1' for i in x])
            if isinstance(x, (list, np.ndarray)) else np.array([-1])
        )
        
        # Map categories to integers for coloring
        unique_cats = set()
        for cats in df['category_labels']:
            unique_cats.update(set(cats))
        unique_cats.discard('-1')
        mapping_labels_int = {c: i+1 for i, c in enumerate(sorted(unique_cats))}
        mapping_labels_int['-1'] = 0
        
        print(f"Found {len(unique_cats)} semantic categories")
        print("Ready for UMAP visualization")
        
    except Exception as e:
        print(f"Error loading UMAP data: {e}")
        import traceback
        traceback.print_exc()
else:
    print("Data not available or dataframe not loaded. Skipping UMAP computation.")


In [ ]:
df['category_labels']


In [ ]:
row.video_representation_path


In [ ]:
if DATA_AVAILABLE and df is not None and 'category_labels' in df.columns:
    try:
        # Load all embeddings for UMAP
        # Load embeddings for each participant
        X_all_list = []
        X_all_sbj_slice = []
        start_idx = 0
        
        for idx, row in df.iterrows():
            try:
                X = load_embeddings(row)
                if X is not None:
                    X_all_list.append(X)
                    end_idx = start_idx + len(X)
                    X_all_sbj_slice.append((start_idx, end_idx))
                    start_idx = end_idx
            except:
                print(f"Warning: Could not load embeddings for {row.get('trigram', 'unknown')}")
                continue
        
        if X_all_list:
            X_all = np.vstack(X_all_list)
            df['X_all_sbj_slice'] = X_all_sbj_slice[:len(df)]
            print(f"Loaded embeddings: X_all shape {X_all.shape}")
            
            # Normalize with scaler, then L2 normalize
            if scaler is not None:
                X_all_scaled = scaler.transform(X_all)
            else:
                X_all_scaled = X_all.copy()
            
            # L2 normalize
            X_all_norm = X_all_scaled / (np.linalg.norm(X_all_scaled, axis=1, keepdims=True) + 1e-8)
            P_norm = P / (np.linalg.norm(P, axis=1, keepdims=True) + 1e-8) if scaler is None else scaler.transform(P)
            P_norm = P_norm / (np.linalg.norm(P_norm, axis=1, keepdims=True) + 1e-8)
            
            # Fit UMAP on joint data (empirical + prototypes)
            print("Fitting UMAP (this may take a minute)...")
            reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42, verbose=True, n_jobs=-1)
            X_concat = np.vstack([X_all_norm, P_norm])
            embedding = reducer.fit_transform(X_concat)
            
            X_emb = embedding[:len(X_all_norm)]
            P_emb = embedding[len(X_all_norm):]
            
            print(f"UMAP complete: X_emb shape {X_emb.shape}, P_emb shape {P_emb.shape}")
            
            # Build prototype-to-category mapping for coloring
            mapping_prototypes_category = {}
            for pidx in range(len(P)):
                prototype_frames = df['pyr_filtered_raw_embedding_labels'].apply(
                    lambda x: np.count_nonzero(np.array(x) == pidx) if isinstance(x, (list, np.ndarray)) else 0
                ).sum()
                if prototype_frames > 0:
                    # Find most common category for this prototype
                    cats = []
                    for _, row in df.iterrows():
                        labels = row['pyr_filtered_raw_embedding_labels']
                        if isinstance(labels, (list, np.ndarray)):
                            frame_cats = [mapping_cluster_category.get(lbl, ['-1'])[0] 
                                         for i, lbl in enumerate(labels) if int(lbl) == pidx]
                            cats.extend(frame_cats)
                    if cats:
                        most_common_cat = Counter(cats).most_common(1)[0][0]
                        mapping_prototypes_category[pidx] = most_common_cat
            
            print(f"Mapped {len(mapping_prototypes_category)} prototypes to categories")
            
        else:
            print("ERROR: Could not load any embeddings")
            
    except Exception as e:
        print(f"Error during UMAP preparation: {e}")
        import traceback
        traceback.print_exc()
else:
    print("Prerequisites not met for UMAP computation.")


In [ ]:
if DATA_AVAILABLE and df is not None and 'X_all_sbj_slice' in df.columns:
    try:
        # Fig 5.1: UMAP scatter plot with prototypes and empirical data
        import random
        
        # Create color map for categories
        unique_cats_umap = sorted(set(mapping_prototypes_category.values()))
        n_cats = len(unique_cats_umap)
        cmap_umap = plt.cm.get_cmap('tab20', n_cats)
        cat_to_color = {cat: cmap_umap(i) for i, cat in enumerate(unique_cats_umap)}
        proto_colors = [cat_to_color.get(mapping_prototypes_category[i], (0.5, 0.5, 0.5, 1.0)) 
                       for i in range(len(P))]
        
        fig, ax = plt.subplots(figsize=(14, 14))
        
        # Plot all empirical data in light grey
        ax.scatter(X_emb[:, 0], X_emb[:, 1], c='lightgrey', s=3, alpha=0.3, label='Empirical data')
        
        # Plot prototypes with colors
        ax.scatter(P_emb[:, 0], P_emb[:, 1], c=proto_colors, s=100, edgecolor='black', 
                  linewidth=0.5, alpha=0.8, label='Prototypes')
        
        # Highlight 2 random participants
        if len(df) > 0:
            sampled_idx = random.sample(range(len(df)), min(2, len(df)))
            highlight_colors = ['dodgerblue', 'darkorange']
            for i, idx in enumerate(sampled_idx):
                start, end = df.iloc[idx]['X_all_sbj_slice']
                x_subj = X_emb[start:end]
                ax.scatter(x_subj[:, 0], x_subj[:, 1], s=15, alpha=0.7, 
                          c=[highlight_colors[i]], label=f'Participant {idx+1}')
        
        ax.set_title('UMAP projection: Prototypes + empirical data', fontsize=14, fontweight='bold', pad=15)
        ax.set_xlabel('UMAP-1', fontsize=12)
        ax.set_ylabel('UMAP-2', fontsize=12)
        ax.legend(loc='best', frameon=True, fontsize=10)
        ax.set_xticks([])
        ax.set_yticks([])
        
        plt.tight_layout()
        fig.savefig(os.path.join(fig_dir, 'fig5_umap_prototypes.png'), dpi=150, bbox_inches='tight')
        plt.show()
        print("Figure saved to fig5_umap_prototypes.png")
        
    except Exception as e:
        print(f"Error plotting UMAP: {e}")
        import traceback
        traceback.print_exc()
else:
    print("Prerequisites not met for UMAP plotting.")


## 3. Chapter 5 — Recursive Prototyping Workflow

Prototype emergence and chronogram visualization with semantic categories.

**Pipeline overview:**
1. Apply pyramid labels (prototype-to-category mapping)
2. Update segmentation from embedding labels (majority voting)
3. Generate full-cohort chronogram
4. Plot prototype prevalence by category

**See:** `smartflat.features.symbolization.utils`, `smartflat.utils.utils_visualization`

In [ ]:
if DATA_AVAILABLE and df is not None and 'pyr_filtered_raw_embedding_labels' not in df.columns:
    try:
        # Apply pyramid labels (maps prototypes to semantic categories)
        print("Applying pyramid labels...")
        df = add_pyramid_labels(df, CONFIG_NAME, ANNOTATOR_ID, ROUND_NUMBER)
        
        # Update segmentation from embedding labels (majority voting over frames)
        print("Updating segmentation from embedding labels...")
        update_cols = [
            'cat_segm_embedding_labels', 'cat_segments_labels',
            'cat_cpts', 'cat_n_cpts', 'cat_segments_has_separations', 'cat_n_segmented', 
            'cat_segments_length', 'cat_n_embed_changes', 'cat_percent_embed_changes',
            'cat_sum_cpts_withdrawn', 'cat_percent_cpts_withdrawn'
        ]
        
        df[update_cols] = df.apply(
            update_segmentation_from_embedding_labels,
            axis=1,
            temporal_segmentation_col='raw_cpts',
            embedding_labels_col='pyr_filtered_raw_embedding_labels',
            combine_func_name='majority_voting',
            filter_noise_labels=False,
            verbose=False
        )
        
        print("Segmentation updated successfully")
        print(f"Dataframe now has {len(df.columns)} columns")
        
    except Exception as e:
        print(f"Error applying pyramid labels: {e}")
        import traceback
        traceback.print_exc()
elif DATA_AVAILABLE and df is not None:
    print("Pyramid labels already applied")


In [ ]:
if DATA_AVAILABLE and df is not None and 'cat_segm_embedding_labels' in df.columns:
    try:
        # Fig 5.2: Full-cohort chronogram with categorical labels
        print("Generating full-cohort chronogram...")
        fig, ax = plt.subplots(figsize=(25, 12))
        
        idx, cmap, norm, boundaries, mapping = plot_chronogames(
            df, 
            labels_col='cat_segm_embedding_labels',
            n_t_max=99999,
            time_calibration='embeddings',
            figsize=(25, 12),
            ax=ax
        )
        
        plt.tight_layout()
        fig.savefig(os.path.join(fig_dir, 'fig5_chronogram_full.png'), dpi=150, bbox_inches='tight')
        plt.show()
        print("Chronogram saved to fig5_chronogram_full.png")
        
    except Exception as e:
        print(f"Error generating chronogram: {e}")
        import traceback
        traceback.print_exc()
else:
    print("Prerequisites not met for chronogram generation.")


In [ ]:
if DATA_AVAILABLE and df is not None and 'cat_segm_embedding_labels' in df.columns:
    try:
        # Fig 5.3: Prototype prevalence by category
        print("Generating prototype prevalence plot...")
        fig, ax = plt.subplots(figsize=(10, 6))
        
        plot_cluster_prevalence(
            df,
            segments_labels_col='cat_segm_embedding_labels',
            title='Prototype prevalence by semantic category',
            figsize=(10, 6),
            ax=ax
        )
        
        plt.tight_layout()
        fig.savefig(os.path.join(fig_dir, 'fig5_prototype_prevalence.png'), dpi=150, bbox_inches='tight')
        plt.show()
        print("Prototype prevalence plot saved")
        
    except Exception as e:
        print(f"Error generating prevalence plot: {e}")
        import traceback
        traceback.print_exc()
else:
    print("Prerequisites not met for prevalence plot.")


## 4. Chapter 5 — Temporal Segmentation

Change-point detection and self-similarity matrix visualization (Gram matrices).

**Pipeline overview:**
1. Select 6 representative participants
2. Load per-participant embeddings
3. Plot Gram matrices (self-similarity X @ X.T) with change-point overlays
4. Generate mini-chronogram for selected participants

**See:** `smartflat.utils.utils_visualization`, `smartflat.utils.utils_dataset`

In [ ]:
if DATA_AVAILABLE and df is not None and 'video_representation_path' in df.columns:
    try:
        # Select 6 representative participants for gram matrices
        SELECTED_TASK_IDS = [12, 6, 103, 11, 55, 79]
        selected_df = df[df['task_number_int'].isin(SELECTED_TASK_IDS)].reset_index(drop=True)
        
        if len(selected_df) > 0:
            print(f"Found {len(selected_df)} participants matching task IDs {SELECTED_TASK_IDS}")
            
            # Create gram matrix output folder
            gram_dir_full = os.path.join(fig_dir, 'gram_matrices')
            os.makedirs(gram_dir_full, exist_ok=True)
            print(f"Gram matrices will be saved to {gram_dir_full}")
            
            # Generate gram matrices for each selected participant
            n_shown = 0
            for idx, row in selected_df.iterrows():
                try:
                    # Load embedding for this participant
                    X = load_embeddings(row)
                    if X is None or len(X) == 0:
                        print(f"Warning: Could not load embeddings for {row.get('trigram', 'unknown')}")
                        continue
                    
                    # Normalize to L2
                    X_norm = normalize_data(X, normalization='l2')
                    
                    # Get change points
                    cpts = row.get('cpts', np.array([]))
                    if not isinstance(cpts, (list, np.ndarray)):
                        cpts = np.array([])
                    else:
                        cpts = np.array(cpts)
                    
                    # Compute Gram matrix (self-similarity)
                    gram = X_norm @ X_norm.T
                    
                    # Fig 5.4a: Gram matrix with change-point lines
                    fig, ax = plt.subplots(figsize=(10, 10))
                    im = ax.imshow(gram, aspect='auto', cmap='coolwarm', interpolation='nearest')
                    
                    # Add change-point vertical and horizontal lines
                    for cpt in cpts:
                        ax.axvline(cpt, color='black', linestyle='-', linewidth=0.5, alpha=0.3)
                        ax.axhline(cpt, color='black', linestyle='-', linewidth=0.5, alpha=0.3)
                    
                    # Convert x/y ticks to minutes (assuming 25 fps)
                    fps = 25
                    frame_duration = 8  # seconds per 8 frames at 25 fps
                    xticks = ax.get_xticks()
                    ax.set_xticklabels([f'{int(t * frame_duration / (fps * 60))}' for t in xticks], fontweight='bold', fontsize=10)
                    yticks = ax.get_yticks()
                    ax.set_yticklabels([f'{int(t * frame_duration / (fps * 60))}' for t in yticks], fontweight='bold', fontsize=10)
                    
                    ax.set_xlabel('Time (minutes)', fontweight='bold')
                    ax.set_ylabel('Time (minutes)', fontweight='bold')
                    ax.set_title(f'{row.trigram} (N={len(X)}, duration={row.duration:.1f}s)', fontweight='bold', fontsize=11)
                    plt.colorbar(im, ax=ax, label='Self-similarity')
                    
                    plt.tight_layout()
                    gram_path = os.path.join(gram_dir_full, f'gram_{row.trigram}_with_cpts.png')
                    fig.savefig(gram_path, dpi=100, bbox_inches='tight')
                    
                    if n_shown < 3:  # Show first 3
                        plt.show()
                        n_shown += 1
                    else:
                        plt.close(fig)
                    
                    print(f"Saved gram matrix for {row.trigram}")
                    
                except Exception as e:
                    print(f"Error processing {row.get('trigram', 'unknown')}: {e}")
                    continue
        else:
            print(f"No participants found with task IDs {SELECTED_TASK_IDS}")
            
    except Exception as e:
        print(f"Error generating gram matrices: {e}")
        import traceback
        traceback.print_exc()
else:
    print("Prerequisites not met for gram matrix generation.")


In [ ]:
if DATA_AVAILABLE and df is not None and 'cat_segm_embedding_labels' in df.columns:
    try:
        # Fig 5.5: Mini-chronogram for 6 selected participants
        print("Generating mini-chronogram for selected participants...")
        SELECTED_TASK_IDS = [12, 6, 103, 11, 55, 79]
        selected_for_chrono = df[df['task_number_int'].isin(SELECTED_TASK_IDS)]
        
        if len(selected_for_chrono) > 0:
            fig, ax = plt.subplots(figsize=(20, 3))
            
            plot_chronogames(
                selected_for_chrono,
                labels_col='cat_segm_embedding_labels',
                n_t_max=9999,
                time_calibration='embeddings',
                figsize=(20, 3),
                ax=ax
            )
            
            plt.tight_layout()
            fig.savefig(os.path.join(fig_dir, 'fig5_chronogram_selected.png'), dpi=150, bbox_inches='tight')
            plt.show()
            print("Mini-chronogram saved to fig5_chronogram_selected.png")
        else:
            print("No selected participants found for mini-chronogram")
            
    except Exception as e:
        print(f"Error generating mini-chronogram: {e}")
        import traceback
        traceback.print_exc()
else:
    print("Prerequisites not met for mini-chronogram generation.")


## 5. Chapter 6 — Barycenter Averaging & Clinical Analysis

Temporal-Wasserstein distance matrices and clinical group comparisons.

**Note:** Chapter 6 figures require precomputed barycenter distance matrices from NB06. The source thesis notebook did not include Ch. 6 figures — they are included here as placeholders for future expansion.

**Pipeline overview:**
1. Load precomputed pairwise TWE distance matrices (from NB06 outputs)
2. Visualize distance matrices by diagnostic group
3. Generate violin plots of distances by diagnosis
4. Clinical group comparisons

**See:** `smartflat.features.symbolic_barycenter.visualization`

In [ ]:
# Load Chapter 6 precomputed outputs from NB06
ch6_ok = False
if DATA_AVAILABLE:
    import pickle
    ch6_dir = os.path.join(DATA_ROOT, 'outputs', 'symbolic_barycenter')
    ch6_files = ['D_twe.npy', 'D_G.npy', 'X_aeon.npy', 'barycenters.pkl', 'split_data.pkl']
    ch6_ok = all(os.path.exists(os.path.join(ch6_dir, f)) for f in ch6_files)

    if ch6_ok:
        D_twe = np.load(os.path.join(ch6_dir, 'D_twe.npy'))
        D_G = np.load(os.path.join(ch6_dir, 'D_G.npy'))
        X_aeon_ch6 = np.load(os.path.join(ch6_dir, 'X_aeon.npy'))
        with open(os.path.join(ch6_dir, 'barycenters.pkl'), 'rb') as f:
            barycenters_ch6 = pickle.load(f)
        with open(os.path.join(ch6_dir, 'split_data.pkl'), 'rb') as f:
            split_data_ch6 = pickle.load(f)

        G_ch6 = D_G.shape[0]
        print(f"Ch.6 data loaded: D_twe {D_twe.shape}, G={G_ch6}")
        print(f"Barycenters: {list(barycenters_ch6.keys())}")
        print(f"Sequences: {X_aeon_ch6.shape}")
    else:
        missing = [f for f in ch6_files if not os.path.exists(os.path.join(ch6_dir, f))]
        print(f"Ch.6 outputs not found. Missing: {missing}")
        print("Run NB06 first to generate symbolic_barycenter outputs.")
else:
    print("Data not available. Skipping Chapter 6 figures.")


In [ ]:
# Fig 6.1: Pairwise TWE distance matrix sorted by diagnostic group
if ch6_ok and df is not None:
    # Sort by pathology group for block structure
    patho_order = {'HEALTHY': 0, 'RIL': 1, 'TBI': 2}
    sort_idx = np.argsort([patho_order.get(p, 3) for p in df['pathologie'].values])
    D_sorted = D_twe[np.ix_(sort_idx, sort_idx)]
    patho_sorted = df['pathologie'].values[sort_idx]

    fig, ax = plt.subplots(figsize=(8, 7))
    im = ax.imshow(D_sorted, cmap='viridis', aspect='equal')
    plt.colorbar(im, ax=ax, label='TW-TWE distance', shrink=0.8)

    # Add group boundary lines
    boundaries = []
    prev = patho_sorted[0]
    for i, p in enumerate(patho_sorted):
        if p != prev:
            boundaries.append(i)
            prev = p
    for b in boundaries:
        ax.axhline(b - 0.5, color='white', linewidth=1.5)
        ax.axvline(b - 0.5, color='white', linewidth=1.5)

    # Group labels
    groups_sorted = list(dict.fromkeys(patho_sorted))
    group_centers = []
    start = 0
    for g in groups_sorted:
        count = (patho_sorted == g).sum()
        group_centers.append(start + count / 2)
        start += count
    ax.set_xticks(group_centers)
    ax.set_xticklabels(groups_sorted, fontweight='bold')
    ax.set_yticks(group_centers)
    ax.set_yticklabels(groups_sorted, fontweight='bold')
    ax.set_title('Pairwise TW-TWE Distance Matrix', fontweight='bold')

    fig.savefig(os.path.join(fig_dir, 'chapter_6_distance_matrix_pairwise.png'),
                dpi=150, bbox_inches='tight')
    print(f"Saved: chapter_6_distance_matrix_pairwise.png")
    plt.show()
else:
    print("Prerequisites not met for distance matrix figure.")


In [ ]:
# Fig 6.2–6.3: DBA barycenter chronograms per group
# chapter_6_non_bg.png = without background symbol (symbol 0 masked)
# chapter_6_bg.png    = all symbols shown
if ch6_ok:
    groups_ch6 = ['HEALTHY', 'RIL', 'TBI']
    cmap_ch6 = plt.cm.get_cmap('tab20', G_ch6)

    for mode, fname in [('bg', 'chapter_6_bg.png'), ('non_bg', 'chapter_6_non_bg.png')]:
        fig, axes = plt.subplots(3, 1, figsize=(20, 4), sharex=True)
        for gi, grp in enumerate(groups_ch6):
            b = barycenters_ch6[grp].squeeze().copy()
            if mode == 'non_bg':
                # Mask background symbol (symbol 0) as NaN for transparency
                b_float = b.astype(float)
                b_float[b == 0] = np.nan
                data = b_float[np.newaxis, :]
            else:
                data = b[np.newaxis, :]
            axes[gi].imshow(data, aspect='auto', cmap=cmap_ch6,
                            vmin=0, vmax=G_ch6 - 1, interpolation='nearest')
            axes[gi].set_yticks([])
            axes[gi].set_ylabel(grp, fontweight='bold', fontsize=10)
        axes[-1].set_xlabel('Time (symbols)', fontsize=10)
        suffix = 'without background' if mode == 'non_bg' else 'with background'
        fig.suptitle(f'DBA Barycenters ({suffix})', fontweight='bold', y=1.02)
        fig.tight_layout()
        fig.savefig(os.path.join(fig_dir, fname), dpi=150, bbox_inches='tight')
        print(f"Saved: {fname}")
        plt.show()
else:
    print("Prerequisites not met for barycenter chronograms.")


In [ ]:
# Fig 6.4: Set-median barycenters in TW-TWE space (medoid per group)
if ch6_ok and df is not None:
    groups_ch6 = ['HEALTHY', 'RIL', 'TBI']
    train_idx_ch6 = split_data_ch6['train_idx']
    patho_train = split_data_ch6['pathologie_train']
    X_symbolic_ch6 = X_aeon_ch6[:, 0, :]
    cmap_ch6 = plt.cm.get_cmap('tab20', G_ch6)

    fig, axes = plt.subplots(3, 1, figsize=(20, 4), sharex=True)
    for gi, grp in enumerate(groups_ch6):
        grp_train = train_idx_ch6[patho_train == grp]
        D_sub = D_twe[np.ix_(grp_train, grp_train)]
        medoid_global = grp_train[np.argmin(D_sub.sum(axis=1))]
        medoid_seq = X_symbolic_ch6[medoid_global]

        axes[gi].imshow(medoid_seq[np.newaxis, :], aspect='auto', cmap=cmap_ch6,
                        vmin=0, vmax=G_ch6 - 1, interpolation='nearest')
        axes[gi].set_yticks([])
        pid = split_data_ch6['participant_ids_train'][patho_train == grp][
            np.argmin(D_sub.sum(axis=1))
        ]
        axes[gi].set_ylabel(f'{grp}\n(P{pid})', fontweight='bold', fontsize=9)

    axes[-1].set_xlabel('Time (symbols)', fontsize=10)
    fig.suptitle('Set-Median Barycenters (Medoid in TW-TWE Space)', fontweight='bold', y=1.02)
    fig.tight_layout()
    fig.savefig(os.path.join(fig_dir, 'chapter_6_barycenters_median_twe.png'),
                dpi=150, bbox_inches='tight')
    print(f"Saved: chapter_6_barycenters_median_twe.png")
    plt.show()
else:
    print("Prerequisites not met for medoid barycenter figure.")


In [ ]:
# Fig 6.5: Classification AUC summary (from NB06b baseline comparison)
if ch6_ok:
    csv_path = os.path.join(ch6_dir, 'baseline_comparison.csv')
    if os.path.exists(csv_path):
        df_comp = pd.read_csv(csv_path)
        auc_summary = (
            df_comp.groupby(['method', 'comparison'])['auc']
            .agg(['mean', 'std']).reset_index()
        )
        method_order = ['tw_twe'] + sorted(
            [m for m in df_comp['method'].unique() if m != 'tw_twe']
        )
        comparison_order = sorted(df_comp['comparison'].unique())

        fig, ax = plt.subplots(figsize=(12, 5))
        n_methods = len(method_order)
        bar_width = 0.8 / n_methods
        colors = plt.cm.Set2(np.linspace(0, 1, n_methods))

        for mi, method in enumerate(method_order):
            means, stds, xs = [], [], []
            for ci, comp in enumerate(comparison_order):
                row = auc_summary[
                    (auc_summary['method'] == method) & (auc_summary['comparison'] == comp)
                ]
                means.append(row['mean'].values[0] if len(row) else 0)
                stds.append(row['std'].values[0] if len(row) else 0)
                xs.append(ci + mi * bar_width)
            ax.bar(xs, means, bar_width, yerr=stds, label=method,
                   color=colors[mi], edgecolor='white', linewidth=0.5, capsize=2)

        ax.set_xticks([ci + (n_methods - 1) * bar_width / 2 for ci in range(len(comparison_order))])
        ax.set_xticklabels([c.replace('_', ' ') for c in comparison_order])
        ax.set_ylabel('AUC-ROC')
        ax.set_title('Classification Performance by Barycenter Method', fontweight='bold')
        ax.axhline(0.5, color='gray', linestyle='--', linewidth=0.8, label='chance')
        ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
        ax.set_ylim(0, 1.05)
        fig.tight_layout()
        fig.savefig(os.path.join(fig_dir, 'chapter_6_classification_report.png'),
                    dpi=150, bbox_inches='tight')
        print(f"Saved: chapter_6_classification_report.png")
        plt.show()
    else:
        print(f"baseline_comparison.csv not found at {csv_path}")
        print("Run NB06b first to generate baseline comparison results.")
else:
    print("Prerequisites not met for classification report figure.")


## 6. Published Paper — Recursive Prototyping (Chapter 5 paper)

Summary of key paper figures generated in sections 2–4.

**Key paper figures:**
- **Fig 1 (Section 2):** UMAP projection of VideoMAE-v2 embeddings with prototypes colored by semantic category
- **Fig 2 (Section 3):** Full-cohort chronogram showing symbolic representation across all 122 participants
- **Fig 3 (Section 3):** Prototype prevalence distribution by semantic category
- **Fig 4 (Section 4):** Gram matrices (self-similarity) for 6 selected participants with change-point overlays
- **Fig 5 (Section 4):** Mini-chronogram of 6 selected participants showing symbolic sequences

**Output location:** All figures saved to `{SMARTFLAT_DATA_ROOT}/figures/thesis/`

In [ ]:
if DATA_AVAILABLE:
    print("="*70)
    print("THESIS FIGURE GENERATION SUMMARY")
    print("="*70)
    print("")
    print("CHAPTER 4 — Study Design")
    print("  ✓ fig4_cohort_breakdown.png — Cohort composition bar chart")
    print("  ✓ fig4_duration_by_group.png — Task duration by diagnostic group")
    print("")
    print("CHAPTER 5 — Recursive Prototyping & Temporal Segmentation")
    print("  ✓ fig5_umap_prototypes.png — UMAP of prototypes + empirical data (Section 2)")
    print("  ✓ fig5_chronogram_full.png — Full-cohort chronogram (Section 3)")
    print("  ✓ fig5_prototype_prevalence.png — Prototype prevalence by category (Section 3)")
    print("  ✓ gram_matrices/gram_*.png — Gram matrices for 6 selected participants (Section 4)")
    print("  ✓ fig5_chronogram_selected.png — Mini-chronogram of 6 selected participants (Section 4)")
    print("")
    print("CHAPTER 6 — Barycenter & Clinical Analysis")
    print("  ✗ Not included in source thesis notebook — placeholder for future expansion")
    print("")
    print("="*70)
    print(f"Output directory: {fig_dir}")
    print("="*70)
else:
    print("Data not available. No figures were generated.")
